# Bank Management System Project

This project demonstrates Object-Oriented Programming (OOP) concepts by creating a simple bank management system with different types of accounts.

## 1. Abstract Class: `Account`

This abstract base class defines the common interface and properties for all types of bank accounts. It includes private attributes for security and abstract methods that must be implemented by concrete subclasses.

In [ ]:
from abc import ABC, abstractmethod

class Account(ABC):
    def __init__(self, account_number, customer_name, initial_balance=0):
        self.__account_number = account_number  # Private attribute
        self.__customer_name = customer_name    # Private attribute
        self.__balance = initial_balance        # Private attribute

    def display_account_info(self):
        print(f"Account Number: {self.__account_number}")
        print(f"Customer Name: {self.__customer_name}")
        print(f"Balance: ${self.__balance:.2f}")

    @abstractmethod
    def withdraw(self, amount):
        pass

    def deposit(self, amount):
        if amount > 0:
            self.__balance += amount
            print(f"Deposited ${amount:.2f}. New balance: ${self.__balance:.2f}")
        else:
            print("Deposit amount must be positive.")

    # Helper methods to access private attributes
    def get_account_number(self):
        return self.__account_number

    def get_balance(self):
        return self.__balance

    def _update_balance(self, new_balance):
        self.__balance = new_balance

## 2. Concrete Class: `SavingsAccount`

This class inherits from `Account` and implements the `withdraw` method with a balance sufficiency check.

In [ ]:
class SavingsAccount(Account):
    def __init__(self, account_number, customer_name, initial_balance=0, interest_rate=0.01):
        super().__init__(account_number, customer_name, initial_balance)
        self.interest_rate = interest_rate

    def withdraw(self, amount):
        if amount <= 0:
            print("Withdrawal amount must be positive.")
            return False
        if self.get_balance() >= amount:
            self._update_balance(self.get_balance() - amount)
            print(f"Withdrew ${amount:.2f}. New balance: ${self.get_balance():.2f}")
            return True
        else:
            print("Insufficient balance for withdrawal.")
            return False

    def apply_interest(self):
        interest = self.get_balance() * self.interest_rate
        self.deposit(interest)
        print(f"Interest of ${interest:.2f} applied.")

## 3. Concrete Class: `CurrentAccount`

This class also inherits from `Account` and implements `withdraw`, but allows withdrawals up to an overdraft limit.

In [ ]:
class CurrentAccount(Account):
    def __init__(self, account_number, customer_name, initial_balance=0, overdraft_limit=1000):
        super().__init__(account_number, customer_name, initial_balance)
        self.overdraft_limit = overdraft_limit

    def withdraw(self, amount):
        if amount <= 0:
            print("Withdrawal amount must be positive.")
            return False
        if self.get_balance() + self.overdraft_limit >= amount:
            self._update_balance(self.get_balance() - amount)
            print(f"Withdrew ${amount:.2f}. New balance: ${self.get_balance():.2f}")
            return True
        else:
            print("Withdrawal exceeds overdraft limit.")
            return False

## 4. Class: `OnlineBanking`

This class provides functionality for online banking operations like login and money transfer.

In [ ]:
class OnlineBanking:
    def __init__(self, username, password):
        self.username = username
        self.password = password
        self.is_logged_in = False

    def login(self, entered_username, entered_password):
        if self.username == entered_username and self.password == entered_password:
            self.is_logged_in = True
            print(f"{self.username} logged in successfully.")
            return True
        else:
            print("Invalid username or password.")
            return False

    def logout(self):
        self.is_logged_in = False
        print(f"{self.username} logged out.")

    def transfer_money(self, from_account, to_account, amount):
        if not self.is_logged_in:
            print("Please log in to transfer money.")
            return False
        if not isinstance(from_account, Account) or not isinstance(to_account, Account):
            print("Both accounts must be valid bank accounts.")
            return False
        if from_account.withdraw(amount):
            to_account.deposit(amount)
            print(f"Transferred ${amount:.2f} from {from_account.get_account_number()} to {to_account.get_account_number()}.")
            return True
        return False

## 5. Class: `SmartAccount` (Multiple Inheritance)

This class combines features of both `SavingsAccount` and `OnlineBanking` using multiple inheritance.

In [ ]:
class SmartAccount(SavingsAccount, OnlineBanking):
    def __init__(self, account_number, customer_name, initial_balance=0, interest_rate=0.01, username, password):
        # Initialize both parent classes using super() with MRO consideration
        super().__init__(account_number, customer_name, initial_balance, interest_rate)
        OnlineBanking.__init__(self, username, password) # Explicitly call the second parent's init

    def display_account_info(self):
        super().display_account_info()
        print(f"Interest Rate: {self.interest_rate*100:.2f}%")
        print(f"Online Username: {self.username}")

## 6. & 7. Object Creation and Polymorphism Demonstration

Creating objects from different account types and demonstrating polymorphism by calling `withdraw()` in a loop.

In [ ]:
# Create multiple objects
sav_acc1 = SavingsAccount("S1001", "Alice Smith", 1500, 0.02)
curr_acc1 = CurrentAccount("C2001", "Bob Johnson", 500, 2000)
smart_acc1 = SmartAccount("SM3001", "Charlie Brown", 2500, 0.015, "charlieB", "pass123")

# Store all account objects in a list
accounts = [sav_acc1, curr_acc1, smart_acc1]

print("\n--- Demonstrating Polymorphism (Withdrawals) ---")
for i, account in enumerate(accounts):
    print(f"\nAttempting withdrawal from Account {i+1} ({account.__class__.__name__})")
    account.display_account_info()
    account.deposit(200) # Ensure some balance for withdrawals
    account.withdraw(500)
    account.withdraw(3000) # This should fail for savings/smart and work for current within limit

print("\n--- Account Information After Polymorphic Withdrawals ---")
for account in accounts:
    account.display_account_info()

## 8. Method Resolution Order (MRO) for `SmartAccount`

The MRO determines the order in which Python searches for methods and attributes in classes that use multiple inheritance. It follows the C3 linearization algorithm.

In [ ]:
print("MRO for SmartAccount:")
print(SmartAccount.__mro__)

print("\nExplanation:\nPython searches for methods in the following order:
1.  SmartAccount itself.
2.  SavingsAccount (the first parent in the inheritance list).
3.  OnlineBanking (the second parent).
4.  Account (parent of SavingsAccount, found via SavingsAccount's MRO).
5.  ABC (parent of Account).
6.  object (the base class of all Python objects).")

## 9. Accessing Private Attributes (Name Mangling)

Private attributes in Python are prefixed with `__` (double underscore). Python performs name mangling on these attributes, making them accessible via `_ClassName__attributeName`.

In [ ]:
print("--- Accessing Private Account Number ---")
sav_acc_test = SavingsAccount("S9999", "Test User", 100)

# Attempt direct access (will raise AttributeError)
try:
    print(f"Direct access attempt: {sav_acc_test.__account_number}")
except AttributeError as e:
    print(f"Error trying direct access: {e}")

# Access using Name Mangling
print(f"Access using Name Mangling: {sav_acc_test._Account__account_number}")

# Access using a public getter method (recommended practice)
print(f"Access using public getter method: {sav_acc_test.get_account_number()}")

## 10. `deposit()` Method in Parent Class

The `deposit()` method was already added to the `Account` parent class, allowing all child classes to inherit and use it without reimplementation.

In [ ]:
print("\n--- Testing deposit() from parent class ---")
sav_acc2 = SavingsAccount("S1002", "Emily White", 500)
curr_acc2 = CurrentAccount("C2002", "David Green", 200)

sav_acc2.display_account_info()
sav_acc2.deposit(100)
sav_acc2.display_account_info()

curr_acc2.display_account_info()
curr_acc2.deposit(300)
curr_acc2.display_account_info()

## 11. Testing the Application

Performing various operations to test the functionality of the bank management system.

In [ ]:
print("\n--- Application Test Scenarios ---")

# 1. Create accounts
print("\nCreating accounts...")
alice_sav = SavingsAccount("S4001", "Alice Wonderland", 1000)
bob_curr = CurrentAccount("C5001", "Bob The Builder", 300, 500)
charlie_smart = SmartAccount("SM6001", "Charlie Chaplin", 2000, 0.02, "charlie_c", "secure_pass")

alice_sav.display_account_info()
bob_curr.display_account_info()
charlie_smart.display_account_info()

# 2. Deposit money
print("\nDepositing money...")
alice_sav.deposit(500)
bob_curr.deposit(1000)
charlie_smart.deposit(200)

# 3. Withdraw money
print("\nWithdrawing money...")
alice_sav.withdraw(200) # Should succeed
bob_curr.withdraw(1600) # Should succeed (uses overdraft)
alice_sav.withdraw(2000) # Should fail (insufficient balance)
charlie_smart.withdraw(1000) # Should succeed (uses SavingsAccount's withdraw)

# 4. Display account information
print("\nDisplaying updated account information...")
alice_sav.display_account_info()
bob_curr.display_account_info()
charlie_smart.display_account_info()

# 5. Log in to online banking
print("\nLogging in to online banking (SmartAccount)...")
charlie_smart.login("charlie_c", "secure_pass")
charlie_smart.login("wrong_user", "bad_pass")

# 6. Transfer money between accounts
print("\nTransferring money...")
if charlie_smart.is_logged_in:
    charlie_smart.transfer_money(charlie_smart, alice_sav, 300) # Smart to Savings
    charlie_smart.transfer_money(alice_sav, bob_curr, 100) # Savings to Current
else:
    print("Cannot transfer: Charlie is not logged in.")

print("\nDisplaying account information after transfer...")
alice_sav.display_account_info()
bob_curr.display_account_info()
charlie_smart.display_account_info()

# Apply interest to Savings and Smart Accounts
print("\nApplying interest...")
alice_sav.apply_interest()
charlie_smart.apply_interest()

print("\nFinal Account Information:")
alice_sav.display_account_info()
bob_curr.display_account_info()
charlie_smart.display_account_info()

### Abstract Base Classes in Python

An Abstract Base Class (ABC) in Python is a class that cannot be instantiated directly. Its primary purpose is to define a blueprint for other classes. It can contain abstract methods (methods that have a declaration but no implementation) and concrete methods (methods with implementation).

To define an ABC, we use the `abc` module and the `@abstractmethod` decorator for abstract methods. Any concrete class inheriting from an ABC must implement all its abstract methods.

In [ ]:
from abc import ABC, abstractmethod

class Shape(ABC):

    @abstractmethod
    def Area(self):
        pass

    @abstractmethod
    def Perimeter(self):
        pass

    def display(self):
        print("This is a shape.")

Now, let's create concrete classes that inherit from `Shape` and implement its abstract methods.

In [ ]:
import math

class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius

    def Area(self):
        return math.pi * self.radius**2

    def Perimeter(self):
        return 2 * math.pi * self.radius

    def display(self):
        print(f"This is a circle with radius {self.radius}.")

In [ ]:
class Rectangle(Shape):
    def __init__(self, length, width):
        self.length = length
        self.width = width

    def Area(self):
        return self.length * self.width

    def Perimeter(self):
        return 2 * (self.length + self.width)

    def display(self):
        print(f"This is a rectangle with length {self.length} and width {self.width}.")

In [ ]:
class Square(Rectangle):
    def __init__(self, side):
        super().__init__(side, side)

    def display(self):
        print(f"This is a square with side {self.length}.")

Finally, let's create objects from each class and demonstrate their methods.

In [ ]:
shapes = [
    Circle(5),
    Rectangle(4, 6),
    Square(7)
]

for shape in shapes:
    print(f"\n--- {shape.__class__.__name__} ---")
    shape.display()
    print(f"Area: {shape.Area():.2f}")
    print(f"Perimeter: {shape.Perimeter():.2f}")